In [1]:
import pandas as pd
import numpy as np
import datetime

# 1. Generate Sample Data (Simulating IoT Sensor Data)
def generate_sensor_data(num_records=100):
    timestamps = pd.date_range(start="2023-10-26", periods=num_records, freq="1min")
    sensor_ids = np.random.choice(["sensor_A", "sensor_B", "sensor_C"], num_records)
    temperatures = np.random.normal(25, 5, num_records)  # Mean 25, std dev 5
    humidities = np.random.uniform(40, 80, num_records)
    data = pd.DataFrame({
        "timestamp": timestamps,
        "sensor_id": sensor_ids,
        "temperature": temperatures,
        "humidity": humidities,
    })
    return data

# 2. Data Cleaning and Preprocessing
def clean_data(df):
    # Handle missing values (replace with mean for simplicity)
    df.fillna(df.mean(numeric_only=True), inplace=True)

    # Remove outliers (simple example: values outside 3 standard deviations)
    for col in ["temperature", "humidity"]:
        mean = df[col].mean()
        std = df[col].std()
        df = df[np.abs(df[col] - mean) <= 3 * std]

    # Convert timestamp to datetime if needed
    if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
        df['timestamp'] = pd.to_datetime(df['timestamp'])

    return df

# 3. Data Aggregation and Transformation //change time
def aggregate_data(df, interval="10min"):
    df.set_index("timestamp", inplace=True) #must set timestamp as index for resample.
    aggregated = df.resample(interval).agg({
        "temperature": "mean",
        "humidity": "mean",
    })
    aggregated["sensor_count"] = df["sensor_id"].resample(interval).nunique() #number of unique sensors in each window.

    return aggregated.reset_index() #reset index to make timestamp a column again.

# 4. Data Storage (Example: CSV) // save in jason
def store_data(df, filename="processed_sensor_data.csv"):
    df.to_csv(filename, index=False)
    print(f"Data stored to {filename}")

# 5. Data Retrieval and Analysis (Example: Basic Analysis)
def analyze_data(filename="processed_sensor_data.csv"):
    try:
        df = pd.read_csv(filename, parse_dates=["timestamp"]) #parse the timestamp column as dates.
        print("\nData Analysis:")
        print(f"Average temperature: {df['temperature'].mean()}")
        print(f"Max humidity: {df['humidity'].max()}")
        print(f"Total sensor count: {df['sensor_count'].sum()}")
    except FileNotFoundError:
        print(f"File {filename} not found.")

# Main Execution
if __name__ == "__main__":
    raw_data = generate_sensor_data()
    cleaned_data = clean_data(raw_data.copy()) # .copy() prevents modifying the original.
    aggregated_data = aggregate_data(cleaned_data.copy())
    store_data(aggregated_data)
    analyze_data()

Data stored to processed_sensor_data.csv

Data Analysis:
Average temperature: 25.558384114740544
Max humidity: 64.13610251236555
Total sensor count: 28
